<a href="https://colab.research.google.com/github/tamaki-gth/cartpole_project/blob/main/cartpole.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

m1=1
m2=1
l=0.5
g=9.8
J=0.5
F=0
tau=0.01

def f(z,m1,m2,l,J,F,g):

    x,x_dot,theta,theta_dot=z

    A1=np.array([[m1+m2,m2*l*np.cos(theta)],[m2*l*np.cos(theta),m2*l**2+J]])
    A2=np.array([[m2*l*theta_dot**2*np.sin(theta)+F],[m2*g*l*np.sin(theta)]])
    A1_inv=np.linalg.inv(A1)

    h=A1_inv@A2

    #return np.array([h[0,0],h[1,0],x_dot,theta_dot])
    return np.array([x_dot, h[0,0], theta_dot, h[1,0]])

def RungeKutta(z,tau,m1,m2,l,J,F,g):
    k1=f(z,m1,m2,l,J,F,g)
    k2=f(z+k1*tau/2,m1,m2,l,J,F,g)
    k3=f(z+k2*tau/2,m1,m2,l,J,F,g)
    k4=f(z+k3*tau,m1,m2,l,J,F,g)
    return z+tau/6*(k1+2*k2+2*k3+k4)


In [2]:
class CartPoleEnv:
    def __init__(self):
        self.reset()

    def reset(self):
        self.z = np.array([0.0, 0.0, 0.05, 0.0])
        return self.z

    def step(self, action):
        F = float(action)

        self.z = RungeKutta(self.z, tau, m1, m2, l, J, F, g)

        x, x_dot, theta, theta_dot = self.z

        reward = -(theta**2 + 0.1*theta_dot**2 + 0.01*x**2 + 0.001*F**2)

        done = abs(theta) > 0.2

        return self.z, reward, done

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal

class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__()

        self.actor = nn.Sequential(
            nn.Linear(4, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh()
        )

        self.mu = nn.Linear(64, 1)
        self.log_std = nn.Parameter(torch.zeros(1))

        self.critic = nn.Sequential(
            nn.Linear(4, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

    def forward(self, state):
        x = self.actor(state)
        mu = self.mu(x)
        std = torch.exp(self.log_std)
        dist = Normal(mu, std)
        value = self.critic(state)
        return dist, value

In [4]:
class PPO:
    def __init__(self):
        self.model = ActorCritic()
        self.optimizer = optim.Adam(self.model.parameters(), lr=3e-4)

        self.gamma = 0.99
        self.eps_clip = 0.2

    def select_action(self, state):
        state = torch.FloatTensor(state)
        dist, value = self.model(state)

        action = dist.sample()
        log_prob = dist.log_prob(action)

        return action.detach().numpy(), log_prob.detach(), value.detach()

    def update(self, memory):
        states = torch.FloatTensor(memory['states'])
        actions = torch.FloatTensor(memory['actions'])
        old_log_probs = torch.stack(memory['log_probs']).detach()
        returns = torch.FloatTensor(memory['returns'])

        for _ in range(10):
            dist, values = self.model(states)
            log_probs = dist.log_prob(actions)

            ratio = torch.exp(log_probs - old_log_probs)

            advantages = returns - values.detach().squeeze()

            surr1 = ratio * advantages
            surr2 = torch.clamp(ratio, 1-self.eps_clip, 1+self.eps_clip) * advantages

            actor_loss = -torch.min(surr1, surr2).mean()
            critic_loss = (returns - values.squeeze()).pow(2).mean()

            loss = actor_loss + 0.5 * critic_loss

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

In [5]:
from IPython.display import HTML
import matplotlib.pyplot as plt
from matplotlib import animation
from google.colab import drive
drive.mount('/content/drive')

def save_animation(z_array,l,actions, filename="cartpole.mp4"):
    x = z_array[:,0]
    x_dot=z_array[:,1]
    theta = z_array[:,2]
    theta_dot=z_array[:,3]

    xc = x + l*np.sin(theta)
    yc = l*np.cos(theta)

    fig, ax = plt.subplots()

    def update(i):
        ax.clear()
        ax.set_xlim(np.min(x)-1, np.max(x)+1)
        ax.set_ylim(-l, l)
        ax.set_aspect('equal')

        ax.plot([x[i]], [0], "bo")
        ax.plot([xc[i]], [yc[i]], "ro")

    ani = animation.FuncAnimation(fig, update, frames=len(z_array))
    filename = "/content/drive/MyDrive/lab_activity/cartpole.mp4"
    ani.save(filename, writer="ffmpeg", fps=60)

    plt.close()
    display(save_animation(states_record, l, filename))

Mounted at /content/drive


In [6]:
import numpy as np

def train():
    env = CartPoleEnv()
    agent = PPO()

    max_episodes = 300
    max_steps = 1000

    max_timesteps = max_episodes * max_steps
    timesteps_so_far = 0

    for episode in range(max_episodes):
        state = env.reset()

        record = (episode == max_episodes - 1)

        if record:
            states_record = []
            actions_record = []

        memory = {'states':[], 'actions':[], 'log_probs':[],
                  'rewards':[], 'dones':[]}

        total_reward = 0

        for t in range(max_steps):
            action, log_prob, _ = agent.select_action(state)

            action_env = action * 10.0

            next_state, reward, done = env.step(action_env)

            if record:
                states_record.append(state.copy())
                actions_record.append(action_env[0])

            memory['states'].append(state)
            memory['actions'].append(action)
            memory['log_probs'].append(log_prob)
            memory['rewards'].append(reward)
            memory['dones'].append(done)

            state = next_state
            total_reward += reward
            timesteps_so_far += 1

            if done:
                break

        returns = []
        G = 0
        for r, d in zip(reversed(memory['rewards']), reversed(memory['dones'])):
            if d:
                G = 0
            G = r + agent.gamma * G
            returns.insert(0, G)

        memory['returns'] = returns

        cur_lrmult = max(1.0 - float(timesteps_so_far) / (max_timesteps/2), 0)
        if cur_lrmult < 1e-5:
            cur_lrmult = 1e-5

        for param_group in agent.optimizer.param_groups:
            param_group['lr'] = 3e-4 * cur_lrmult

        agent.update(memory)

        print(f"Episode {episode} | Reward {total_reward:.2f}")

        if record:
            save_animation(np.array(states_record), l, np.array(actions_record))


train()

/tmp/ipykernel_5695/3659431897.py:10: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  F = float(action)
/tmp/ipykernel_5695/3643655931.py:19: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  states = torch.FloatTensor(memory['states'])


Episode 0 | Reward -7.60
Episode 1 | Reward -6.98
Episode 2 | Reward -5.38
Episode 3 | Reward -11.54
Episode 4 | Reward -29.93
Episode 5 | Reward -10.74
Episode 6 | Reward -7.32
Episode 7 | Reward -5.37
Episode 8 | Reward -9.03
Episode 9 | Reward -8.78
Episode 10 | Reward -8.97
Episode 11 | Reward -4.87
Episode 12 | Reward -5.99
Episode 13 | Reward -8.96
Episode 14 | Reward -5.21
Episode 15 | Reward -6.17
Episode 16 | Reward -6.54
Episode 17 | Reward -4.46
Episode 18 | Reward -5.36
Episode 19 | Reward -4.05
Episode 20 | Reward -5.66
Episode 21 | Reward -5.70
Episode 22 | Reward -5.55
Episode 23 | Reward -4.93
Episode 24 | Reward -8.35
Episode 25 | Reward -6.44
Episode 26 | Reward -4.56
Episode 27 | Reward -7.09
Episode 28 | Reward -4.50
Episode 29 | Reward -5.45
Episode 30 | Reward -4.80
Episode 31 | Reward -5.81
Episode 32 | Reward -6.49
Episode 33 | Reward -6.63
Episode 34 | Reward -8.50
Episode 35 | Reward -14.44
Episode 36 | Reward -7.78
Episode 37 | Reward -8.83
Episode 38 | Rewar

NameError: name 'states_record' is not defined